In [53]:
#Data manipulation
import pandas as pd
import numpy as np

# Plotting
import matplotlib.pyplot as plt

# Preprocessing
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
train_test_split,
RandomizedSearchCV,
TimeSeriesSplit,
cross_val_score
)

# K-mean clustering
from sklearn.cluster import KMeans

# Classifier
from sklearn.ensemble import RandomForestClassifier

# Metrics
from sklearn.metrics import (precision_recall_curve,
roc_curve,
RocCurveDisplay,
ConfusionMatrixDisplay
)

from sklearn.metrics import (accuracy_score,
f1_score,
recall_score,
precision_score,
roc_auc_score,
auc
)

from sklearn.metrics import (classification_report,
confusion_matrix
)

# Import tensorflow and keras
import tensorflow
import keras 

# Import pandas-ta
import pandas_ta as ta

In [54]:
files = ["CMA1.csv", "HML1.csv", "MKT1.csv", "RMW1.csv", "SMB1.csv", "UMD1.csv"]
tickers = ["cma", "hml", "mkt", "rmw", "smb", "umd",]

data = pd.DataFrame()

# Load files, extract the 'Close' column, and concatenate DataFrames
for i, file in enumerate(files):
    df = pd.read_csv(file, index_col=0, parse_dates=True, usecols=['date', 'close'])
    df.columns = [f"Close_{tickers[i]}"]  # Rename 'Close' column with ticker suffix
    data = pd.concat([data, df], axis=1)

# Also save each original ticker data into a single dataframe for later calculation of technical indicators
for ticker in tickers:
    globals()[ticker] = pd.read_csv(ticker + ".csv", index_col=0, parse_dates=True)

# Save the concatenated DataFrame to a new CSV file
data.to_csv("concatenated_closes1.csv")

data


,Close_cma,Close_hml,Close_mkt,Close_rmw,Close_smb,Close_umd
date,,,,,,
2001-07-31,100.377846,102.897835,92.047866,102.887173,110.192425,103.156118
2001-08-31,103.451999,106.861916,103.911195,99.858324,102.773984,104.308992
2001-09-30,95.626899,103.151944,85.606645,113.447956,102.409075,116.495149
2001-10-31,99.397393,109.108447,84.773921,104.620226,109.586239,119.208716
2001-11-30,102.660913,110.189952,93.368129,101.106723,104.856177,113.454421
...,...,...,...,...,...,...
2024-01-31,301.100885,405.031702,654.070987,141.519633,156.635419,1138.914871
2024-02-29,314.086095,414.159101,660.496602,136.806863,155.040184,1142.557471
2024-03-31,310.378491,412.436252,665.841389,137.146367,157.106535,1124.838603


In [55]:
data.describe().T

,count,mean,std,min,25%,50%,75%,max
Close_cma,275.0,184.168091,50.825932,95.626899,143.172760,170.023283,210.569827,314.406001
Close_hml,275.0,270.056293,116.484796,102.897835,143.934213,276.324789,383.113150,491.996100
Close_mkt,275.0,463.040281,225.227283,84.773921,242.425269,520.460543,665.091726,793.804146
Close_rmw,275.0,123.789403,15.408153,81.991585,114.569213,125.783191,135.155130,166.687863
Close_smb,275.0,135.373373,18.835174,102.409075,121.275390,130.167542,154.409678,194.713366
Close_umd,275.0,538.352864,291.748493,103.156118,311.376258,513.109975,753.631157,1196.496681


In [56]:
# check for missing values
data.isnull().sum()

Close_cma    0
Close_hml    0
Close_mkt    0
Close_rmw    0
Close_smb    0
Close_umd    0
dtype: int64

In [57]:
files = ["CMA1.csv", "HML1.csv", "MKT1.csv", "RMW1.csv", "SMB1.csv", "UMD1.csv"]
tickers = ["cma", "hml", "mkt", "rmw", "smb", "umd"]

ret = pd.DataFrame()

# Load files, extract the 'RET' column, and concatenate DataFrames
for i, file in enumerate(files):
    df = pd.read_csv(file, index_col=0, parse_dates=True, usecols=['date','return'])
    df.columns = [f"RET_{tickers[i]}"]  # Rename 'RET' column with ticker suffix
    ret = pd.concat([ret, df], axis=1)

# Save the concatenated DataFrame to a new CSV file if needed
ret.to_csv("concatenated_returns1.csv")

ret

,RET_cma,RET_hml,RET_mkt,RET_rmw,RET_smb,RET_umd
date,,,,,,
2001-07-31,0.003778,0.028978,-0.079521,0.028872,0.101924,0.031561
2001-08-31,0.030626,0.038524,0.128882,-0.029439,-0.067323,0.011176
2001-09-30,-0.075640,-0.034717,-0.176156,0.136089,-0.003551,0.116827
2001-10-31,0.039429,0.057745,-0.009727,-0.077813,0.070083,0.023293
2001-11-30,0.032833,0.009912,0.101378,-0.033583,-0.043163,-0.048271
...,...,...,...,...,...,...
2024-01-31,0.005302,0.012470,-0.038249,-0.008550,0.037862,0.009134
2024-02-29,0.043126,0.022535,0.009824,-0.033301,-0.010184,0.003198
2024-03-31,-0.011804,-0.004160,0.008092,0.002482,0.013328,-0.015508


In [58]:
# Define the SMA periods
sma_periods = [5, 10, 20, 50, 100, 200]

# Loop through each ticker and calculate SMA for each period
for ticker in tickers:
    # Identify the column that contains the close prices (assuming the column name is 'close' followed by the ticker)
    close_col = f"Close_{ticker}"
    
    # Check if the column exists in the data DataFrame
    if close_col in data.columns:
        for period in sma_periods:
            # Calculate SMA
            sma_col = f"SMA_{period}_{ticker}"
            sma_series = ta.sma(data[close_col], length=period)
            
            # Add SMA to the DataFrame with appropriate name
            data[sma_col] = sma_series
    else:
        print(f"Column {close_col} does not exist in the DataFrame.")

print(data)

             Close_cma   Close_hml   Close_mkt   Close_rmw   Close_smb  \
date                                                                     
2001-07-31  100.377846  102.897835   92.047866  102.887173  110.192425   
2001-08-31  103.451999  106.861916  103.911195   99.858324  102.773984   
2001-09-30   95.626899  103.151944   85.606645  113.447956  102.409075   
2001-10-31   99.397393  109.108447   84.773921  104.620226  109.586239   
2001-11-30  102.660913  110.189952   93.368129  101.106723  104.856177   
...                ...         ...         ...         ...         ...   
2024-01-31  301.100885  405.031702  654.070987  141.519633  156.635419   
2024-02-29  314.086095  414.159101  660.496602  136.806863  155.040184   
2024-03-31  310.378491  412.436252  665.841389  137.146367  157.106535   
2024-04-30  314.406001  412.964811  664.575750  136.504217  156.261798   
2024-05-31  309.313253  425.790703  654.957220  137.938773  159.092561   

              Close_umd   SMA_5_cma  

In [59]:
# Define the EMA periods
ema_periods = [5, 8, 10, 12, 25, 26, 50, 200]

# Loop through each ticker and calculate EMA for each period
for ticker in tickers:
    # Identify the column that contains the close prices (assuming the column name is 'close' followed by the ticker)
    close_col = f"Close_{ticker}"
    
    # Check if the column exists in the data DataFrame
    if close_col in data.columns:
        for period in ema_periods:
            # Calculate EMA
            ema_col = f"EMA_{period}_{ticker}"
            ema_series = ta.ema(data[close_col], length=period)
            
            # Add EMA to the DataFrame with appropriate name
            data[ema_col] = ema_series
    else:
        print(f"Column {close_col} does not exist in the DataFrame.")

print(data) 

             Close_cma   Close_hml   Close_mkt   Close_rmw   Close_smb  \
date                                                                     
2001-07-31  100.377846  102.897835   92.047866  102.887173  110.192425   
2001-08-31  103.451999  106.861916  103.911195   99.858324  102.773984   
2001-09-30   95.626899  103.151944   85.606645  113.447956  102.409075   
2001-10-31   99.397393  109.108447   84.773921  104.620226  109.586239   
2001-11-30  102.660913  110.189952   93.368129  101.106723  104.856177   
...                ...         ...         ...         ...         ...   
2024-01-31  301.100885  405.031702  654.070987  141.519633  156.635419   
2024-02-29  314.086095  414.159101  660.496602  136.806863  155.040184   
2024-03-31  310.378491  412.436252  665.841389  137.146367  157.106535   
2024-04-30  314.406001  412.964811  664.575750  136.504217  156.261798   
2024-05-31  309.313253  425.790703  654.957220  137.938773  159.092561   

              Close_umd   SMA_5_cma  

In [60]:
# Define the RSI periods
rsi_periods = [5, 9, 14, 30]

# Loop through each ticker and calculate RSI for each period
for ticker in tickers:
    # Identify the column that contains the close prices (assuming the column name is 'close' followed by the ticker)
    close_col = f"Close_{ticker}"
    
    # Check if the column exists in the data DataFrame
    if close_col in data.columns:
        for period in rsi_periods:
            # Calculate RSI
            rsi_col = f"RSI_{period}_{ticker}"
            rsi_series = ta.rsi(data[close_col], length=period)
            
            # Add RSI to the DataFrame with appropriate name
            data[rsi_col] = rsi_series
    else:
        print(f"Column {close_col} does not exist in the DataFrame.")

print(data)

             Close_cma   Close_hml   Close_mkt   Close_rmw   Close_smb  \
date                                                                     
2001-07-31  100.377846  102.897835   92.047866  102.887173  110.192425   
2001-08-31  103.451999  106.861916  103.911195   99.858324  102.773984   
2001-09-30   95.626899  103.151944   85.606645  113.447956  102.409075   
2001-10-31   99.397393  109.108447   84.773921  104.620226  109.586239   
2001-11-30  102.660913  110.189952   93.368129  101.106723  104.856177   
...                ...         ...         ...         ...         ...   
2024-01-31  301.100885  405.031702  654.070987  141.519633  156.635419   
2024-02-29  314.086095  414.159101  660.496602  136.806863  155.040184   
2024-03-31  310.378491  412.436252  665.841389  137.146367  157.106535   
2024-04-30  314.406001  412.964811  664.575750  136.504217  156.261798   
2024-05-31  309.313253  425.790703  654.957220  137.938773  159.092561   

              Close_umd   SMA_5_cma  

C:\Users\tonfo\AppData\Local\Temp\ipykernel_2776\957566562.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[rsi_col] = rsi_series
C:\Users\tonfo\AppData\Local\Temp\ipykernel_2776\957566562.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[rsi_col] = rsi_series
C:\Users\tonfo\AppData\Local\Temp\ipykernel_2776\957566562.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using 

In [61]:
# Define the Bollinger Bands parameters
bb_params = [
    (10, 1.5),
    (20, 2),
    (50, 2.5)
]

# Loop through each ticker and calculate Bollinger Bands for each parameter set
for ticker in tickers:
    # Identify the column that contains the close prices (assuming the column name is 'close' followed by the ticker)
    close_col = f"Close_{ticker}"
    
    # Check if the column exists in the data DataFrame
    if close_col in data.columns:
        for period, std_dev in bb_params:
            # Calculate Bollinger Bands
            bb = ta.bbands(data[close_col], length=period, std=std_dev)
            
            # Extract the Bollinger Bands components from the returned DataFrame
            bb_upper_col = bb.columns[0]
            bb_middle_col = bb.columns[1]
            bb_lower_col = bb.columns[2]
            
            bb_upper = bb[bb_upper_col]
            bb_middle = bb[bb_middle_col]
            bb_lower = bb[bb_lower_col]
            
            # Add Bollinger Bands components to the DataFrame with appropriate names
            data[f"BB_upper_{period}_{std_dev}_{ticker}"] = bb_upper
            data[f"BB_middle_{period}_{std_dev}_{ticker}"] = bb_middle
            data[f"BB_lower_{period}_{std_dev}_{ticker}"] = bb_lower
    else:
        print(f"Column {close_col} does not exist in the DataFrame.")

print(data)

             Close_cma   Close_hml   Close_mkt   Close_rmw   Close_smb  \
date                                                                     
2001-07-31  100.377846  102.897835   92.047866  102.887173  110.192425   
2001-08-31  103.451999  106.861916  103.911195   99.858324  102.773984   
2001-09-30   95.626899  103.151944   85.606645  113.447956  102.409075   
2001-10-31   99.397393  109.108447   84.773921  104.620226  109.586239   
2001-11-30  102.660913  110.189952   93.368129  101.106723  104.856177   
...                ...         ...         ...         ...         ...   
2024-01-31  301.100885  405.031702  654.070987  141.519633  156.635419   
2024-02-29  314.086095  414.159101  660.496602  136.806863  155.040184   
2024-03-31  310.378491  412.436252  665.841389  137.146367  157.106535   
2024-04-30  314.406001  412.964811  664.575750  136.504217  156.261798   
2024-05-31  309.313253  425.790703  654.957220  137.938773  159.092561   

              Close_umd   SMA_5_cma  

C:\Users\tonfo\AppData\Local\Temp\ipykernel_2776\3031231226.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[f"BB_upper_{period}_{std_dev}_{ticker}"] = bb_upper
C:\Users\tonfo\AppData\Local\Temp\ipykernel_2776\3031231226.py:30: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[f"BB_middle_{period}_{std_dev}_{ticker}"] = bb_middle
C:\Users\tonfo\AppData\Local\Temp\ipykernel_2776\3031231226.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which h

In [62]:
# Build the feature matrix
feat = pd.concat([ret, data], axis=1) # Feature matrix

# Save the matrix
feat.to_csv('feat.csv')

In [63]:
df = pd.read_csv('cma_model.csv')

# Define target
df['Target'] = np.where(df['cma 6m'] > np.log(1.05) * df['return 6m'], 1, 0)

# Drop the last row where there is no label
#df = df.iloc[:-1]

# Drop the rows with NaN value
df.dropna(inplace=True)

X = df.drop(['Target', 'date'],axis=1)
y = df['Target']


In [64]:
#check data
X.to_csv('X_cma_test.csv')

In [65]:
X_train = X[-600:]

# Output the training data size
print(f"Train Size: {len(X_train)}")

Train Size: 600


In [66]:
# Use StandardScaler
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), index=X_train.index, columns=X_train.columns)

X_train_scaled.describe()

,RET_cma,RET_hml,RET_mkt,RET_rmw,RET_smb,RET_umd,Close_cma,Close_hml,Close_mkt,Close_rmw,...,BB_middle_10_1.5_umd,BB_lower_10_1.5_umd,BB_upper_20_2_umd,BB_middle_20_2_umd,BB_lower_20_2_umd,BB_upper_50_2.5_umd,BB_middle_50_2.5_umd,BB_lower_50_2.5_umd,cma 6m,return 6m
count,6.000000e+02,6.000000e+02,6.000000e+02,6.000000e+02,6.000000e+02,6.000000e+02,6.000000e+02,6.000000e+02,6.000000e+02,6.000000e+02,...,6.000000e+02,6.000000e+02,6.000000e+02,6.000000e+02,6.000000e+02,6.000000e+02,6.000000e+02,6.000000e+02,6.000000e+02,6.000000e+02
mean,-1.628327e-17,-2.368476e-17,-1.776357e-17,-1.184238e-17,-1.184238e-17,2.220446e-17,6.631732e-16,1.326346e-15,2.084259e-15,2.842171e-15,...,2.557954e-15,9.355479e-16,1.444770e-15,1.894781e-16,-1.983598e-15,-1.326346e-15,-1.752672e-15,-1.598721e-15,1.421085e-16,-5.210647e-16
std,1.000834e+00,1.000834e+00,1.000834e+00,1.000834e+00,1.000834e+00,1.000834e+00,1.000834e+00,1.000834e+00,1.000834e+00,1.000834e+00,...,1.000834e+00,1.000834e+00,1.000834e+00,1.000834e+00,1.000834e+00,1.000834e+00,1.000834e+00,1.000834e+00,1.000834e+00,1.000834e+00
min,-3.944142e+00,-3.213913e+00,-4.252402e+00,-3.830438e+00,-4.353315e+00,-3.824744e+00,-1.829507e+00,-1.690341e+00,-3.037193e+00,-1.496836e+00,...,-1.580556e+00,-1.661580e+00,-1.630313e+00,-1.638278e+00,-1.753664e+00,-2.936268e+00,-1.804259e+00,-1.513043e+00,-3.036525e+00,-2.600597e+00
25%,-5.795812e-01,-6.444107e-01,-5.618135e-01,-6.139951e-01,-5.439792e-01,-6.010392e-01,-7.582066e-01,-9.327851e-01,-6.347992e-01,-7.311128e-01,...,-7.057882e-01,-7.033379e-01,-7.037865e-01,-7.074572e-01,-7.665194e-01,-4.289320e-01,-6.728156e-01,-8.195556e-01,-4.905502e-01,-7.641238e-01
50%,-3.611497e-02,-3.127872e-02,4.014638e-03,-3.274661e-02,2.924788e-02,-1.346630e-02,-1.722841e-01,1.904705e-02,9.786714e-02,-3.388916e-01,...,-3.250388e-01,-3.018140e-01,-2.170784e-01,-2.716522e-01,-3.037307e-01,-1.026061e-02,-2.843335e-01,-2.052881e-01,6.063540e-02,9.954914e-02
75%,5.381297e-01,6.233493e-01,5.655585e-01,6.159390e-01,6.342119e-01,5.501374e-01,6.137171e-01,8.430991e-01,7.340521e-01,7.610768e-01,...,5.397128e-01,5.412301e-01,4.802905e-01,6.211224e-01,6.894694e-01,4.177976e-01,6.951344e-01,8.549206e-01,7.476685e-01,7.605978e-01
max,6.227381e+00,4.378179e+00,3.731070e+00,4.789656e+00,3.463224e+00,4.118957e+00,2.177712e+00,2.096365e+00,1.925124e+00,2.620698e+00,...,3.362127e+00,3.223444e+00,3.666484e+00,3.439676e+00,3.132211e+00,2.694835e+00,3.370634e+00,3.141188e+00,1.756699e+00,2.330181e+00


In [67]:
# Store the scaled data in the original variables
X_train = X_train_scaled

In [68]:
df = X_train

columns_to_keep = list(df.columns[:6])  # Create a list from the slice
#columns_to_keep.extend(df.columns[-2:])
columns_to_keep.extend(['EMA_25_mkt', 'SMA_100_cma',
                      'EMA_26_rmw', 'EMA_25_smb', 'EMA_25_umd',
                      'EMA_12_cma', 'RSI_9_cma', 'EMA_26_hml',
                      'RSI_9_rmw', 'RSI_9_mkt', 'RSI_9_smb',
                      'RSI_14_hml'])

X_train = X_train[columns_to_keep]

In [69]:
# EDA on the final input matrix
X_train.describe().T

,count,mean,std,min,25%,50%,75%,max
RET_cma,600.0,-1.628327e-17,1.000834,-3.944142,-0.579581,-0.036115,0.538130,6.227381
RET_hml,600.0,-2.368476e-17,1.000834,-3.213913,-0.644411,-0.031279,0.623349,4.378179
RET_mkt,600.0,-1.776357e-17,1.000834,-4.252402,-0.561814,0.004015,0.565558,3.731070
RET_rmw,600.0,-1.184238e-17,1.000834,-3.830438,-0.613995,-0.032747,0.615939,4.789656
RET_smb,600.0,-1.184238e-17,1.000834,-4.353315,-0.543979,0.029248,0.634212,3.463224
RET_umd,600.0,2.220446e-17,1.000834,-3.824744,-0.601039,-0.013466,0.550137,4.118957
EMA_25_mkt,600.0,-2.463215e-15,1.000834,-2.824081,-0.621532,0.103463,0.757130,1.698928
SMA_100_cma,600.0,-1.373716e-15,1.000834,-2.030264,-0.738132,-0.163618,0.603157,2.269808
EMA_26_rmw,600.0,8.052818e-16,1.000834,-1.188578,-0.796401,-0.330899,0.720629,2.478224
EMA_25_smb,600.0,1.302662e-15,1.000834,-3.440241,-0.075135,0.245469,0.556892,1.209195


In [70]:
X_train.to_csv('X_new.csv')
X_train = pd.read_csv('X_new.csv')

In [71]:
# Convert to numpy arrays
X_train = X_train.values

In [72]:
from tensorflow.keras.models import load_model

# Load the model from the H5 file
model = load_model('SL_model_CMA.h5')
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ LSTM1 (LSTM)                    │ (None, 20)             │         3,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dropout1 (Dropout)              │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output (Dense)                  │ (None, 1)              │            21 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,143 (12.28 KB)

 Trainable params: 3,141 (12.27 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [87]:
from tensorflow.keras.models import load_model

# Load the model from the H5 file
model = load_model('SL_model_CMA.h5')

# Load and prepare the input data
X_new = pd.read_csv('X_new.csv').values  # Ensure it's a numpy array

# Make predictions
predictions = model.predict(X_new)

# Print or process the predictions
print(predictions)

ValueError: Exception encountered when calling Sequential.call().

[1mInvalid input shape for input Tensor("data:0", shape=(32, 19), dtype=float32). Expected shape (None, 10, 18), but input has incompatible shape (32, 19)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(32, 19), dtype=float32)
  • training=False
  • mask=None

In [88]:
from tensorflow.keras.models import load_model
import numpy as np
import pandas as pd

# Load the model from the H5 file
model = load_model('SL_model_CMA.h5')

# Load and prepare the input data
X_new = pd.read_csv('X_new.csv').values  # Ensure it's a numpy array

# Check the original shape of X_new
print("Original shape of X_new:", X_new.shape)

# Calculate the total number of elements in X_new
total_elements = np.prod(X_new.shape)
print("Total elements in X_new:", total_elements)

# Determine the appropriate shape for (batch_size, lookback, features)
lookback = 10
features = X_new.shape[1]
batch_size = X_new.shape[0] // lookback

# Check if the number of samples is evenly divisible by lookback
if X_new.shape[0] % lookback != 0:
    raise ValueError("The number of samples is not divisible by the lookback period.")

# Reshape X_new to (batch_size, lookback, features)
X_new = np.reshape(X_new, (batch_size, lookback, features))

# Print the new shape of X_new to confirm
print("New shape of X_new:", X_new.shape)

# Make predictions
predictions = model.predict(X_new)

# Print the prediction formatted to two decimal places
print(f'Test Prediction: {predictions[0][0]:.2f}')


Original shape of X_new: (600, 19)
Total elements in X_new: 11400
New shape of X_new: (60, 10, 19)


ValueError: Exception encountered when calling LSTMCell.call().

[1mDimensions must be equal, but are 19 and 18 for '{{node sequential_1/LSTM1_1/lstm_cell_1/MatMul}} = MatMul[T=DT_FLOAT, grad_a=false, grad_b=false, transpose_a=false, transpose_b=false](sequential_1/LSTM1_1/strided_slice_1, sequential_1/LSTM1_1/lstm_cell_1/Cast/ReadVariableOp)' with input shapes: [32,19], [18,80].[0m

Arguments received by LSTMCell.call():
  • inputs=tf.Tensor(shape=(32, 19), dtype=float32)
  • states=('tf.Tensor(shape=(32, 20), dtype=float32)', 'tf.Tensor(shape=(32, 20), dtype=float32)')
  • training=False

In [ ]:
# Print the prediction formatted to two decimal places
print(f'Test Prediction: {predictions[0][0]:.2f}')